In [20]:
import numpy as np
from scipy.fft import dct, idct, fft, ifft

np.random.seed(42)

In [21]:
def custom_dct(x: np.ndarray):
    """
    1. Reorder input
    2. Perform FFT
    3. Apply phase correction
    """
    N = len(x)
    y = np.zeros(N)

    for n in range(0, N//2):
        y[n] = x[2*n]

    for n in range(N//2, N):
        y[n] = x[2*N - 2*n - 1]

    Y = fft(y)

    """
    From Makhoul's paper on DCN, Eq (10): C(k) = 2Re [ W_{2N}^{k/2} \\sum{n=0}^{N-1} x(n) W_{2N}^{nk} ]
    where W_M = e^{-j2pi/M}. This expands to:    2Re [ e^{-j kpi/2N} \\sum{n=0}^{N-1} x(n) e^{-j 2nkpi/N} ]
    From DFT, Y_k = \\sum_{n=0}^{N-1} x(n) e^{-j 2nkpi/N}.
    When combined:                               2Re [ e^{-j kpi/2N} * Y_k ]

    e^{-j kpi/2N} * Y_k can be written as: e^{-j theta} * Y_k
    e^{-j theta} * Y_k = (cos theta - jsin theta) * (Yr + jYi) = 
    = (Yr * cos theta + Yi sin theta) + j(Yi cos theta - Yr sin theta)
    From 2Re [ e^{-j kpi/2N} * Y_k ], the imaginary part is discarded.

    In full: the final X becomes
    C(k) = 2 [ Y_k_Re * cos theta + Y_k_Im * sin theta ]
    """
    X = np.zeros(N)
    for k in range(N):
        theta = np.pi*k / (2*N)
        X[k] = 2*( Y[k].real * np.cos(theta) + Y[k].imag * np.sin(theta) )
    
    return X

In [22]:
def custom_idct(X: np.ndarray):
    N = len(X)
    x = np.zeros(N)

    for n in range(N):
        x[n] = 0
        partial_sum = 0

        for k in range(1, N):
            theta = np.pi * ((2*n) + 1) * k / (2*N) # constant
            partial_sum += X[k] * np.cos(theta)

        x[n] = 1/N * ( (X[0]/2) + partial_sum )
    
    return x

In [ ]:
def custom_idct_fft(X: np.ndarray):
    """
    1. Calculate V(k)
    2. iFFT
    3. Reorder output
    """
    N = len(X)
    k = np.arange(N)

    X_mirror = np.zeros(N)
    X_mirror[1:] = X[:0:-1]

    V = 0.5 * np.exp( 1j * k * np.pi / (2*N) ) * (X[k] - 1j * X_mirror)
    v = ifft(V).real

    x = np.zeros(N)
    x[0::2] = v[: (N + 1) // 2]
    x[1::2] = v[N - N//2 :][::-1]
    return x

In [24]:
x = np.random.rand(24)
x

array([0.37454012, 0.95071431, 0.73199394, 0.59865848, 0.15601864,
       0.15599452, 0.05808361, 0.86617615, 0.60111501, 0.70807258,
       0.02058449, 0.96990985, 0.83244264, 0.21233911, 0.18182497,
       0.18340451, 0.30424224, 0.52475643, 0.43194502, 0.29122914,
       0.61185289, 0.13949386, 0.29214465, 0.36636184])

In [ ]:
X = dct(x)
np.isclose(X, custom_dct(x)).all()

np.True_

In [35]:
y = idct(X)
print(np.isclose(y, custom_idct(X)).all())
print(np.isclose(y, custom_idct_fft(X)).all())

True
True
